## 00 — The Grid Idea

In Module 03, we built a viewport culling function that scans every feature and checks its bounding box against the viewport. It works — but it scans **every feature, every time**.

For 20,000 features that is fast enough on a modern machine. But:
- What if we had 200,000 features?
- What if culling ran 30 times per second during a smooth pan animation?
- What if we needed to support older hardware?

This notebook introduces a structure that avoids the full scan: the **uniform grid index**.

## The Linear Scan Problem

The `cull()` function from Module 03 has this shape:

```python
def cull(features, viewport_bbox):
    return [f for f in features if bbox_intersects(feature_bbox(f), viewport_bbox)]
```

Every call touches every feature — regardless of where the viewport is. If the user is looking at a 2° × 2° box over Paris, we still evaluate all 20,000 features, including every railroad in Australia, South America, and China.

This is **O(n)** per query. The cost grows linearly with the number of features, and there is no way to skip the irrelevant ones.

## The Grid Idea

The solution is to **pre-organize features by location** so that a viewport query only touches the features that are plausibly nearby.

The simplest version is a **uniform grid**:

1. Divide the world into a regular grid of cells (e.g., 10° × 10°).
2. At build time, assign each feature to every cell whose bbox it overlaps.
3. At query time, find which cells the viewport overlaps — then return only the features from those cells.

```
Build (once):          Query (many times):

┌──┬──┬──┬──┐          ┌──┬──┬──┬──┐
│  │▓▓│  │  │          │  │██│██│  │  <- only check these cells
├──┼──┼──┼──┤          ├──┼──┼──┼──┤
│  │▓▓│▓▓│  │          │  │██│██│  │
├──┼──┼──┼──┤          ├──┼──┼──┼──┤
│  │  │  │  │          │  │  │  │  │
└──┴──┴──┴──┘          └──┴──┴──┴──┘
▓ = feature cells      █ = viewport cells
```

If the viewport covers 2 cells in a 36×18 grid (648 cells total), we only check the features assigned to those 2 cells — a tiny fraction of the total.

## Visualizing the Grid

Let's draw the world grid over the railroad dataset to make the structure concrete.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CELL_SIZE = 10.0  # degrees

fig, ax = plt.subplots(figsize=(14, 7))

# Draw the grid
for lon in range(-180, 181, int(CELL_SIZE)):
    ax.axvline(lon, color='#cccccc', linewidth=0.5)
for lat in range(-90, 91, int(CELL_SIZE)):
    ax.axhline(lat, color='#cccccc', linewidth=0.5)

# Highlight one example viewport
vp = patches.Rectangle((-5, 42), 15, 18,  # roughly France/Western Europe
    linewidth=2, edgecolor='#0066cc', facecolor='#0066cc', alpha=0.15, label='viewport')
ax.add_patch(vp)

# Highlight the cells the viewport touches
for col_start in range(-10, 11, int(CELL_SIZE)):
    for row_start in range(40, 61, int(CELL_SIZE)):
        cell = patches.Rectangle((col_start, row_start), CELL_SIZE, CELL_SIZE,
            linewidth=1.5, edgecolor='#0066cc', facecolor='#0066cc', alpha=0.1)
        ax.add_patch(cell)

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'World grid at {int(CELL_SIZE)}° cells (36 × 18 = 648 cells)\nBlue = cells touched by viewport')
plt.tight_layout()
plt.show()

## How Many Cells Does a Viewport Touch?

For a grid of 10° cells, how many cells does each zoom level's viewport overlap?

In [ ]:
import math

CELL_SIZE = 10.0

def cells_for_bbox(bbox, cell_size=CELL_SIZE):
    """Return a list of (col, row) cell coordinates that a bbox overlaps."""
    lon_min, lat_min, lon_max, lat_max = bbox
    col_min = int((lon_min + 180) / cell_size)
    col_max = int((lon_max + 180) / cell_size)
    row_min = int((lat_min +  90) / cell_size)
    row_max = int((lat_max +  90) / cell_size)
    return [
        (col, row)
        for col in range(col_min, col_max + 1)
        for row in range(row_min, row_max + 1)
    ]

viewports = [
    ("World zoom 2",       [-180, -90,  180,  90]),
    ("Europe zoom 5",      [  -10,  35,   40,  70]),
    ("France zoom 7",      [   -5,  42,   10,  52]),
    ("Paris zoom 10",      [    1,  48,    4,  50]),
    ("Central Paris z13",  [  2.3, 48.8, 2.4, 48.9]),
]

total_cells = (360 // int(CELL_SIZE)) * (180 // int(CELL_SIZE))

print(f"Grid: {int(CELL_SIZE)}° cells  |  {total_cells} total cells\n")
print(f"{'Viewport':<22} {'Cells hit':>10} {'% of grid':>11}")
print("-" * 47)

for name, vp in viewports:
    cells = cells_for_bbox(vp)
    pct   = len(cells) / total_cells * 100
    print(f"{name:<22} {len(cells):>10} {pct:>10.1f}%")

At city zoom, the viewport touches a single cell — 0.15% of the grid. Instead of scanning 20,000 features, we only look at the features in that one cell.

## The Tradeoff — Cell Size

Cell size is the only design parameter, and it creates a direct tradeoff:

| Cell size | Cells in grid | Build cost | Query cells hit | Features per cell |
|-----------|--------------|------------|-----------------|-------------------|
| 30° | 72 | Fast | Many | Many |
| 10° | 648 | Moderate | Moderate | Moderate |
| 1° | 64,800 | Slow | Few | Few |
| 0.1° | 6,480,000 | Very slow | Very few | Very few |

A cell that is too large contains almost all features — no speedup over linear scan.

A cell that is too small takes a long time to build and wastes memory on nearly-empty cells.

The right cell size depends on the dataset density and the typical query size (viewport at most common zoom levels). For a world railroad dataset queried at zoom 5–10, **10° cells** is a reasonable starting point.

## Exercise A

Modify `cells_for_bbox` to work with a cell size of `5.0` instead of `10.0`. Then rerun the viewport table above.

How does the number of cells hit change? Does a smaller cell size always help query speed? Explain.

In [ ]:
cell_size = 5.0
total_cells = int(360 / cell_size) * int(180 / cell_size)

print(f"Grid: {cell_size:.1f}° cells  |  {total_cells} total cells")
print()
print(f"{'Viewport':<22} {'Cells hit':>10} {'% of grid':>11}")
print('-' * 47)

for name, vp in viewports:
    cells = cells_for_bbox(vp, cell_size=cell_size)
    pct = len(cells) / total_cells * 100
    print(f"{name:<22} {len(cells):>10} {pct:>10.1f}%")


## Exercise B

A feature with a very large bounding box — one that spans many degrees — will be assigned to many cells.

Load the fine LOD file, compute `cells_for_bbox` for every feature with a 10° grid, and find:
1. The feature assigned to the **most cells**. How many cells does it occupy?
2. The **average number of cells** per feature.

In [ ]:
import json
from pathlib import Path

def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

with open(Path('../../data/lod/railroads_fine.geojson')) as f:
    fine = json.load(f)

def cells_spanned(feature, cell_size=10.0):
    return cells_for_bbox(feature_bbox(feature), cell_size=cell_size)

best_idx, best_feature = max(
    enumerate(fine['features']),
    key=lambda item: len(cells_spanned(item[1]))
)
best_cells = cells_spanned(best_feature)
props = best_feature['properties']

print(f'Feature index: {best_idx}')
print(f"rwdb_rr_id: {props.get('rwdb_rr_id')}")
print(f"category: {props.get('category')}")
print(f'Max cells spanned at 10°: {len(best_cells)}')
print(f'Cells: {best_cells}')
print(f"bbox: {[round(v, 3) for v in feature_bbox(best_feature)]}")


## Check Your Understanding

The grid index is built **once at startup** and then queried many times.

What happens if the underlying feature data changes — for example, if a LOD file is updated or a new feature is added? Does the grid need to be rebuilt, and what does that cost?

Write a 2–3 sentence answer.

If the underlying data changes, the existing grid index becomes stale and either needs to be rebuilt or updated incrementally. In this simple implementation the practical answer is to rebuild it, which is an O(n) startup-style cost, but that is acceptable because the railroad files are static and queried many more times than they are modified.


## Next

In [01 — Building the Index](./01-Building_the_Index.ipynb), we implement the full grid structure and populate it with the LOD features.